# 🐔 Poultry Disease Classification from Fecal Images — FINAL (Corrected & Deployable)
### Transfer-learning benchmark + **poultry-vs-other-animal rejection gate** (novelty) + deployable web app

> **Veterinary disclaimer.** Research prototype for decision support. It does **not** replace PCR, culture or a veterinarian.

## What changed versus the previous notebook (all verified by code review / smoke test)
| # | Problem in old notebook | Effect | Fix |
|---|---|---|---|
| 1 | Augmentation ran **after** `preprocess_input`; Keras `RandomContrast` clips to `[0,255]` | Destroys negative values of MobileNetV2 / DenseNet / ResNet inputs → train/test distribution mismatch | Augment on raw 0-255 pixels, **then** preprocess |
| 2 | Phase-2 `ModelCheckpoint` restarted from `inf` | Fine-tuning could **overwrite a better phase-1 checkpoint** | `initial_value_threshold` + reload best weights before fine-tuning |
| 3 | Fine-tuned only the last 30 layers (≈5 % of DenseNet169) | Under-adaptation | Unfreeze a *fraction* (30 %) of the backbone, BN frozen |
| 4 | `best_model.save(dir)` | **Crashes in Keras 3** | Save `.keras` + `model.export()` |
| 5 | Grad-CAM cell never called the heat-map function (and `get_layer` fails on a nested backbone) | No heat-maps produced | Working nested Grad-CAM (shared with the app) |
| 6 | "Zero leakage" was *asserted*, never computed; only exact-MD5 duplicates were looked at | Possible optimistic metrics | MD5 **and** perceptual-hash near-duplicate audit, group-aware split |
| 7 | `decode_jpeg` only; PNG allowed in extraction | Crash on PNG | `decode_image` |
| 8 | `__import__('scikit-learn')`, `'pillow'` always fail | Pointless pip calls | Correct module names |
| 9 | JPEG-compression robustness claimed but not implemented; no reliability diagram | Report/code mismatch | Implemented |
| 10 | Hard-coded "6,812 images" in plots/report while splits sum to 6,494 | Misleading figures | All counts computed |
| 11 | No way to reject non-poultry images; Gradio demo not deployable | — | Gate + `deploy` bundle + Hugging Face / Docker |

## Pipeline
1 Environment → 2 Config → 3 Data ingestion/audit → 4 Leakage-free split → 5 `tf.data` → 6 Training (resumable) →
7 Benchmark & test → 8 Optional *paper-style (leaky) protocol* comparison → 9 Explainability/robustness →
**10 Novelty: poultry vs other-animal gate** → 11 Deployment bundle & QA → 12 Report

In [ ]:
# ==============================================================================
# CELL 1: Environment verification, hardware audit and package setup
# ==============================================================================
import sys, os, platform, subprocess, shutil, importlib, time, json, gc, random, datetime, warnings
warnings.filterwarnings('ignore')

SMOKE_TEST = os.environ.get('POULTRY_SMOKE', '0') == '1'   # synthetic dry-run (developers only)
IN_COLAB = 'google.colab' in sys.modules or 'COLAB_GPU' in os.environ

print('=' * 78); print('SYSTEM & HARDWARE ENVIRONMENT AUDIT'); print('=' * 78)
print(f'OS            : {platform.platform()}')
print(f'Python        : {sys.version.split()[0]}')
print(f'Colab         : {IN_COLAB} | Smoke test: {SMOKE_TEST}')

# pip-name -> import-name (the old notebook used wrong names for scikit-learn / pillow)
PACKAGES = {'numpy': 'numpy', 'pandas': 'pandas', 'matplotlib': 'matplotlib', 'seaborn': 'seaborn',
            'scikit-learn': 'sklearn', 'pillow': 'PIL', 'tqdm': 'tqdm', 'tabulate': 'tabulate',
            'opencv-python-headless': 'cv2', 'psutil': 'psutil'}
for pip_name, mod in PACKAGES.items():
    try:
        importlib.import_module(mod)
    except ImportError:
        print(f'  [INSTALLING] {pip_name}')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name])

import psutil
vm = psutil.virtual_memory()
print(f'RAM           : {vm.total/2**30:.1f} GB total | {vm.available/2**30:.1f} GB free')
_d = shutil.disk_usage('/content' if os.path.exists('/content') else '.')
print(f'Disk          : {_d.free/2**30:.1f} GB free')

if SMOKE_TEST:
    import matplotlib; matplotlib.use('Agg')

import numpy as np, tensorflow as tf, keras
print(f'TensorFlow    : {tf.__version__} | Keras {keras.__version__}')
assert int(keras.__version__.split('.')[0]) >= 3, 'Keras 3 (TensorFlow >= 2.16) is required by this notebook.'
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for g in gpus:
        try: tf.config.experimental.set_memory_growth(g, True)
        except Exception as e: print('memory growth note:', e)
    print(f'GPU           : {[g.name for g in gpus]}')
else:
    print('[WARNING] No GPU. In Colab: Runtime > Change runtime type > T4 GPU.')
print('=' * 78)

In [ ]:
# ==============================================================================
# CELL 2: Experiment configuration, directories and reproducibility
# ==============================================================================
import os, json, random, datetime
import numpy as np, tensorflow as tf, keras

BASE = '/tmp/poultry_smoke' if SMOKE_TEST else '/content'
OUT_ROOT = f'{BASE}/results' if SMOKE_TEST else '/content/drive/MyDrive/Poultry_Disease_Classification_Results_FINAL'   # new folder: never mixes with checkpoints from the old (buggy-augmentation) notebook
DRIVE_ROOT = f'{BASE}/drive' if SMOKE_TEST else '/content/drive/MyDrive'

CONFIG = {
    'project_name': 'Poultry_Disease_Classification_TransferLearning_FINAL',
    'random_seed': 42,
    'timestamp': datetime.datetime.now().strftime('%Y%m%d_%H%M%S'),
    'smoke_test': SMOKE_TEST,

    'class_names': ['Coccidiosis', 'Healthy', 'Newcastle_Disease', 'Salmonellosis'],
    'num_classes': 4,

    # ---- paths -----------------------------------------------------------------
    'drive_mount_point': '/content/drive',
    'dataset_folder_name': 'IDSE_Dataset',
    'local_workspace': f'{BASE}/poultry_workspace',
    'local_extracted_data': f'{BASE}/poultry_workspace/extracted_dataset',
    'experiment_output_root': OUT_ROOT,
    'bundle_dir': f'{BASE}/poultry_workspace/deploy_app',
    # OPTIONAL evaluation-only folder of non-poultry images (see Stage 10). Not used for training.
    'non_poultry_dir': f'{DRIVE_ROOT}/non_poultry_eval',

    # ---- data ------------------------------------------------------------------
    'image_size': (96, 96) if SMOKE_TEST else (224, 224),
    'batch_size': 16 if SMOKE_TEST else 32,
    'split_mode': 'auto',        # 'auto' | 'existing' | 'rebuild'   (see Stage 4)
    'split_csv_name': 'friends_split_merged.csv',
    'near_dup_max_hamming': 3,   # perceptual-hash distance (of 64 bits) treated as "same image"
    'train_ratio': 0.70, 'val_ratio': 0.15, 'test_ratio': 0.15,

    # ---- training --------------------------------------------------------------
    'weights': None if SMOKE_TEST else 'imagenet',
    'epochs_head': 1 if SMOKE_TEST else 25,
    'epochs_finetune': 1 if SMOKE_TEST else 20,
    'lr_head': 1e-3,
    'lr_finetune': 3e-5,
    'min_lr': 1e-7,
    'finetune_fraction': 0.30,   # fraction of backbone layers (from the top) that are unfrozen
    'early_stopping_patience': 7,
    'reduce_lr_patience': 3,
    'reduce_lr_factor': 0.5,
    'l2_regularization': 1e-4,
    'dropout_rate': 0.30,
    'architectures': ['MobileNetV2', 'DenseNet169', 'EfficientNetB0', 'ResNet50'],

    # ---- decision / gates ------------------------------------------------------
    'confidence_threshold': 0.75,
    'gate_accept_rate': 0.975,   # each gate keeps >=97.5 % of genuine poultry validation images
    'enable_clip': (not SMOKE_TEST),
    'clip_model': 'openai/clip-vit-base-patch32',

    # ---- optional extras -------------------------------------------------------
    'run_paper_style_comparison': SMOKE_TEST,   # trains one extra model on a *leaky* split (Stage 8)
    'launch_demo': (not SMOKE_TEST),            # temporary public Gradio link at the very end
    'push_to_hf_space': None,                   # e.g. 'your-username/poultry-screening'  (needs HF_TOKEN secret)
}

def enforce_reproducibility(seed=42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)
    keras.utils.set_random_seed(seed)
    print(f'[*] Seeds fixed (seed={seed}). NOTE: bit-exact GPU determinism is not enforced; '
          'expect +/-0.3-0.5 % run-to-run variation.')
enforce_reproducibility(CONFIG['random_seed'])

for d in [CONFIG['local_workspace'], CONFIG['local_extracted_data'], CONFIG['bundle_dir'],
          os.path.join(CONFIG['local_workspace'], 'manifests')]:
    os.makedirs(d, exist_ok=True)
print(f"EXPERIMENT: {CONFIG['project_name']}  | run {CONFIG['timestamp']}")

In [ ]:
# ==============================================================================
# CELL 3: Write the deployable core module (shared by notebook QA and the web app)
# ==============================================================================
CORE_SRC = r'''"""poultry_core.py -- shared inference core for the Poultry Fecal-Image Screening system.

Used by BOTH the training notebook (for gate calibration, Grad-CAM and QA) and the
deployed web app, so training-time and serving-time preprocessing cannot drift apart.

Pipeline for one uploaded image
  1. Gate A (optional): CLIP zero-shot "is this poultry droppings?"   (needs NO extra dataset)
  2. Classifier: 4-class transfer-learning CNN (Coccidiosis/Healthy/Newcastle/Salmonellosis)
  3. Gate B: Mahalanobis distance in the CNN feature space (fitted on poultry data only)
  4. Gate C: softmax-confidence rejection
Research prototype only -- not a substitute for laboratory diagnosis.
"""
from __future__ import annotations

import json
import os
import threading
import warnings

import numpy as np
from PIL import Image, ImageOps

warnings.filterwarnings("ignore", message="The structure of `inputs`")

SOURCE_SIZE = 256  # the training images were distributed pre-resized to 256x256
DEFAULT_CLASS_NAMES = ["Coccidiosis", "Healthy", "Newcastle_Disease", "Salmonellosis"]
DISCLAIMER = (
    "Research prototype for decision support only. It does not replace laboratory "
    "confirmation (PCR, culture, oocyst count) or a veterinarian's judgement."
)

VERDICT_PREDICTION = "PREDICTION"
VERDICT_UNCERTAIN = "UNCERTAIN"
VERDICT_NOT_POULTRY = "NOT_POULTRY_FECES"
VERDICT_OOD = "OUT_OF_DISTRIBUTION"

# ----------------------------------------------------------------------------------
# CLIP zero-shot prompt bank (no training data needed)
# ----------------------------------------------------------------------------------
CLIP_POULTRY = [
    "chicken droppings", "poultry feces on a farm floor",
    "chicken manure", "fresh droppings in a chicken coop",
]
CLIP_OTHER_FECES = [
    "cow dung", "dog feces", "cat feces", "horse manure", "sheep droppings",
    "goat droppings", "pig manure", "human stool", "rabbit droppings",
    "elephant dung", "wild bird droppings on a pavement",
]
CLIP_NOT_FECES = [
    "a live chicken", "a person", "food on a plate", "a landscape", "a building",
    "soil and mud", "a plant with leaves", "a screenshot of text", "a car", "a pet animal",
]


def clip_prompts():
    out = [("poultry", f"a photo of {p}") for p in CLIP_POULTRY]
    out += [("other_feces", f"a photo of {p}") for p in CLIP_OTHER_FECES]
    out += [("not_feces", f"a photo of {p}") for p in CLIP_NOT_FECES]
    return out


# ----------------------------------------------------------------------------------
# Preprocessing (must match the training pipeline exactly)
# ----------------------------------------------------------------------------------
def get_preprocess(arch: str):
    import keras
    table = {
        "MobileNetV2": keras.applications.mobilenet_v2.preprocess_input,
        "DenseNet169": keras.applications.densenet.preprocess_input,
        "EfficientNetB0": keras.applications.efficientnet.preprocess_input,
        "ResNet50": keras.applications.resnet50.preprocess_input,
    }
    return table[arch]


def load_rgb(img) -> Image.Image:
    """Accepts a path, PIL image or numpy array; applies EXIF rotation; returns RGB PIL."""
    if isinstance(img, (str, os.PathLike)):
        img = Image.open(img)
    elif isinstance(img, np.ndarray):
        img = Image.fromarray(img.astype("uint8"))
    img = ImageOps.exif_transpose(img)
    return img.convert("RGB")


def to_model_array(img, size=(224, 224)) -> np.ndarray:
    """Phone photo / dataset image -> float32 array in the 0..255 range, shape (H, W, 3).

    Mirrors training: (dataset is 256x256) -> tf.image.resize bilinear -> round to uint8.
    """
    import tensorflow as tf
    img = load_rgb(img).resize((SOURCE_SIZE, SOURCE_SIZE), Image.LANCZOS)
    arr = np.asarray(img, dtype=np.float32)
    out = tf.image.resize(arr, size, method="bilinear").numpy()
    return np.clip(np.round(out), 0, 255).astype(np.float32)


# ----------------------------------------------------------------------------------
# Threshold helpers (calibrated on IN-distribution data only)
# ----------------------------------------------------------------------------------
def threshold_accept_high(scores_in, accept_rate):
    """Accept if score >= t, so that `accept_rate` of in-distribution samples pass."""
    return float(np.quantile(np.asarray(scores_in, dtype=np.float64), 1.0 - accept_rate))


def threshold_accept_low(scores_in, accept_rate):
    """Accept if score <= t, so that `accept_rate` of in-distribution samples pass."""
    return float(np.quantile(np.asarray(scores_in, dtype=np.float64), accept_rate))


# ----------------------------------------------------------------------------------
# Gate B: Mahalanobis distance on penultimate CNN features
# ----------------------------------------------------------------------------------
class MahalanobisGate:
    """Class-conditional Gaussians with a shared (Ledoit-Wolf shrunk) covariance."""

    def __init__(self, means: np.ndarray, precision: np.ndarray):
        self.means = np.asarray(means, dtype=np.float64)
        self.precision = np.asarray(precision, dtype=np.float64)

    @classmethod
    def fit(cls, feats, labels, num_classes):
        from sklearn.covariance import LedoitWolf
        feats = np.asarray(feats, dtype=np.float64)
        labels = np.asarray(labels)
        means = np.stack([feats[labels == c].mean(axis=0) for c in range(num_classes)])
        centered = feats - means[labels]
        precision = LedoitWolf().fit(centered).precision_
        return cls(means, precision)

    def score(self, feats) -> np.ndarray:
        """Minimum (over classes) squared Mahalanobis distance / D.  Larger = more unfamiliar."""
        feats = np.asarray(feats, dtype=np.float64)
        d = feats.shape[1]
        best = None
        for mu in self.means:
            diff = feats - mu
            dist = ((diff @ self.precision) * diff).sum(axis=1) / d
            best = dist if best is None else np.minimum(best, dist)
        return best

    def save(self, path):
        np.savez_compressed(path, means=self.means.astype(np.float32),
                            precision=self.precision.astype(np.float32))

    @classmethod
    def load(cls, path):
        z = np.load(path)
        return cls(z["means"], z["precision"])


# ----------------------------------------------------------------------------------
# Gate A: CLIP zero-shot "poultry droppings vs everything else"
# ----------------------------------------------------------------------------------
class ClipGate:
    def __init__(self, model_name="openai/clip-vit-base-patch32", device=None):
        import torch
        from transformers import CLIPModel, CLIPProcessor
        self.torch = torch
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = CLIPModel.from_pretrained(model_name).to(self.device).eval()
        self.processor = CLIPProcessor.from_pretrained(model_name)
        prompts = clip_prompts()
        self.groups = np.array([g for g, _ in prompts])
        self.texts = [t for _, t in prompts]
        tok = self.processor.tokenizer(self.texts, padding=True, return_tensors="pt")
        self.input_ids = tok["input_ids"].to(self.device)
        self.attention_mask = tok["attention_mask"].to(self.device)
        self.poultry_mask = self.groups == "poultry"

    def score(self, images, batch_size=32):
        """Returns (p_poultry[N], top_prompt[N]).  p_poultry = softmax mass on poultry prompts."""
        p_out, top_out = [], []
        for i in range(0, len(images), batch_size):
            batch = [load_rgb(x) for x in images[i:i + batch_size]]
            pix = self.processor.image_processor(batch, return_tensors="pt")["pixel_values"]
            with self.torch.no_grad():
                logits = self.model(input_ids=self.input_ids,
                                    attention_mask=self.attention_mask,
                                    pixel_values=pix.to(self.device)).logits_per_image
                probs = logits.softmax(dim=-1).cpu().numpy()
            p_out.extend(probs[:, self.poultry_mask].sum(axis=1).tolist())
            top_out.extend(self.texts[j].replace("a photo of ", "") for j in probs.argmax(axis=1))
        return np.array(p_out), top_out


# ----------------------------------------------------------------------------------
# Grad-CAM (works with the nested-backbone model built in the notebook)
# ----------------------------------------------------------------------------------
def gradcam_heatmap(model, x, class_index=None):
    """x: preprocessed batch (1,H,W,3). Uses the backbone's final feature map."""
    import keras
    import tensorflow as tf
    base = next((l for l in model.layers if isinstance(l, keras.Model)), None)
    if base is None:
        raise ValueError("No nested backbone found for Grad-CAM.")
    head = model.layers[model.layers.index(base) + 1:]
    x = tf.convert_to_tensor(x, dtype=tf.float32)
    with tf.GradientTape() as tape:
        fmap = base(x, training=False)
        tape.watch(fmap)
        h = fmap
        for layer in head:
            try:
                h = layer(h, training=False)
            except TypeError:
                h = layer(h)
        if class_index is None:
            class_index = int(tf.argmax(h[0]))
        target = h[:, class_index]
    grads = tape.gradient(target, fmap)
    weights = tf.reduce_mean(grads, axis=(1, 2))
    cam = tf.reduce_sum(fmap * weights[:, None, None, :], axis=-1)[0]
    cam = tf.nn.relu(cam)
    cam = cam / (tf.reduce_max(cam) + 1e-8)
    return cam.numpy(), class_index


def _jet(v):
    v = np.clip(v, 0.0, 1.0)
    r = np.clip(1.5 - np.abs(4 * v - 3), 0, 1)
    g = np.clip(1.5 - np.abs(4 * v - 2), 0, 1)
    b = np.clip(1.5 - np.abs(4 * v - 1), 0, 1)
    return np.stack([r, g, b], axis=-1)


def overlay_cam(pil_img: Image.Image, cam: np.ndarray, alpha=0.45, out_size=(448, 448)) -> Image.Image:
    base = load_rgb(pil_img).resize(out_size, Image.BILINEAR)
    cam_img = Image.fromarray((cam * 255).astype("uint8")).resize(out_size, Image.BICUBIC)
    heat = _jet(np.asarray(cam_img, dtype=np.float32) / 255.0) * 255.0
    blend = (1 - alpha) * np.asarray(base, dtype=np.float32) + alpha * heat
    return Image.fromarray(np.clip(blend, 0, 255).astype("uint8"))


# ----------------------------------------------------------------------------------
# The analyzer used by the web app and by the notebook QA cells
# ----------------------------------------------------------------------------------
class PoultryAnalyzer:
    def __init__(self, bundle_dir, use_clip=None):
        import keras
        self.bundle_dir = bundle_dir
        with open(os.path.join(bundle_dir, "meta.json")) as f:
            self.meta = json.load(f)
        self.class_names = self.meta.get("class_names", DEFAULT_CLASS_NAMES)
        self.arch = self.meta["arch"]
        self.size = tuple(self.meta.get("image_size", [224, 224]))
        self.conf_thr = float(self.meta.get("conf_threshold", 0.75))
        self.preprocess = get_preprocess(self.arch)
        self.model = keras.models.load_model(os.path.join(bundle_dir, "model.keras"), compile=False)
        feat_out = self.model.get_layer(self.meta.get("feature_layer", "gap")).output
        self.multi = keras.Model(self.model.inputs, [feat_out, self.model.output])
        self.lock = threading.Lock()
        self.warnings = []

        self.maha, self.maha_thr = None, None
        mpath = os.path.join(bundle_dir, "maha.npz")
        if os.path.isfile(mpath) and "maha" in self.meta:
            self.maha = MahalanobisGate.load(mpath)
            self.maha_thr = float(self.meta["maha"]["threshold"])

        self.clip, self.clip_thr = None, None
        cmeta = self.meta.get("clip", {})
        want = cmeta.get("enabled", False) if use_clip is None else use_clip
        if os.environ.get("USE_CLIP") == "0":
            want = False
        if want:
            try:
                self.clip = ClipGate(cmeta.get("model_name", "openai/clip-vit-base-patch32"))
                self.clip_thr = float(cmeta["threshold"])
            except Exception as e:  # noqa: BLE001 - degrade gracefully
                self.warnings.append(
                    f"CLIP gate unavailable ({type(e).__name__}). Non-poultry rejection "
                    "falls back to feature-space outlier detection only.")

    # -- batch scoring -------------------------------------------------------------
    def _prepare(self, images):
        arrs = [to_model_array(im, self.size) for im in images]
        return np.stack([self.preprocess(a.copy()) for a in arrs]).astype("float32")

    def score_batch(self, images, batch_size=32):
        """Raw scores for a list of images (paths / PIL / arrays)."""
        probs, feats = [], []
        for i in range(0, len(images), batch_size):
            x = self._prepare(images[i:i + batch_size])
            with self.lock:
                f, p = self.multi(x, training=False)
            feats.append(np.asarray(f))
            probs.append(np.asarray(p))
        probs, feats = np.concatenate(probs), np.concatenate(feats)
        out = {"probs": probs,
               "maha": self.maha.score(feats) if self.maha is not None else None,
               "clip_p": None, "clip_top": None}
        if self.clip is not None:
            out["clip_p"], out["clip_top"] = self.clip.score(images)
        return out

    # -- decision logic ------------------------------------------------------------
    def decide(self, probs_row, maha_s=None, clip_p=None, clip_top=None):
        top = int(np.argmax(probs_row))
        conf = float(probs_row[top])
        info = {
            "predicted_class": self.class_names[top],
            "confidence": round(conf, 4),
            "confidence_threshold": self.conf_thr,
            "clip_p_poultry": None if clip_p is None else round(float(clip_p), 4),
            "clip_threshold": None if self.clip_thr is None else round(self.clip_thr, 4),
            "clip_best_match": clip_top,
            "feature_outlier_score": None if maha_s is None else round(float(maha_s), 4),
            "feature_outlier_threshold": None if self.maha_thr is None else round(self.maha_thr, 4),
        }
        if clip_p is not None and clip_p < self.clip_thr:
            return VERDICT_NOT_POULTRY, info
        if maha_s is not None and maha_s > self.maha_thr:
            return VERDICT_OOD, info
        if conf < self.conf_thr:
            return VERDICT_UNCERTAIN, info
        return VERDICT_PREDICTION, info

    def analyze(self, image, want_cam=True):
        pil = load_rgb(image)
        x = self._prepare([pil])
        with self.lock:
            f, p = self.multi(x, training=False)
        probs = np.asarray(p)[0]
        maha_s = float(self.maha.score(np.asarray(f))[0]) if self.maha is not None else None
        clip_p = clip_top = None
        if self.clip is not None:
            cp, ct = self.clip.score([pil])
            clip_p, clip_top = float(cp[0]), ct[0]
        verdict, info = self.decide(probs, maha_s, clip_p, clip_top)
        cam_img = None
        if want_cam and verdict in (VERDICT_PREDICTION, VERDICT_UNCERTAIN):
            try:
                with self.lock:
                    cam, _ = gradcam_heatmap(self.model, x)
                cam_img = overlay_cam(pil, cam)
            except Exception as e:  # noqa: BLE001
                self.warnings.append(f"Grad-CAM failed: {type(e).__name__}")
        return {"verdict": verdict, "info": info,
                "probs": {c: float(probs[i]) for i, c in enumerate(self.class_names)},
                "cam": cam_img}

    @staticmethod
    def pretty(name):
        return name.replace("_", " ")

    def message(self, res):
        v, info = res["verdict"], res["info"]
        if v == VERDICT_NOT_POULTRY:
            return ("### Not recognised as poultry droppings\n"
                    f"The image looks more like **{info['clip_best_match']}**. "
                    "Please upload a close-up, well-lit photo of chicken droppings.\n\n"
                    f"_{DISCLAIMER}_")
        if v == VERDICT_OOD:
            return ("### Image is unfamiliar to the model\n"
                    "It does not resemble the poultry-dropping photos the classifier was trained on "
                    "(possible other animal, different subject, or unusual lighting). "
                    "No diagnosis is given.\n\n" f"_{DISCLAIMER}_")
        name = self.pretty(info["predicted_class"])
        if v == VERDICT_UNCERTAIN:
            return (f"### Uncertain - leading guess: {name} ({info['confidence']*100:.1f}%)\n"
                    f"Confidence is below the {self.conf_thr*100:.0f}% reliability threshold. "
                    "Retake the photo or consult a veterinarian.\n\n" f"_{DISCLAIMER}_")
        return (f"### Screening result: **{name}** ({info['confidence']*100:.1f}% confidence)\n"
                f"_{DISCLAIMER}_")
'''
APP_SRC = r'''"""Gradio web app for the Poultry Fecal-Image Screening system.

Run locally :  python app.py
Hugging Face:  push this folder as a Gradio Space (see README.md)
Env vars    :  PORT (default 7860), USE_CLIP=0 to disable the CLIP gate (lighter / faster start)
"""
import os

from poultry_core import DISCLAIMER, PoultryAnalyzer

HERE = os.path.dirname(os.path.abspath(__file__))
BUNDLE_DIR = os.path.join(HERE, "model_bundle")

analyzer = PoultryAnalyzer(BUNDLE_DIR)


def run(img):
    if img is None:
        return "Please upload an image first.", None, None, None
    res = analyzer.analyze(img)
    msg = analyzer.message(res)
    if analyzer.warnings:
        msg += "\n\n> Note: " + " ".join(sorted(set(analyzer.warnings)))
    show_probs = res["verdict"] in ("PREDICTION", "UNCERTAIN")
    probs = {analyzer.pretty(k): v for k, v in res["probs"].items()} if show_probs else None
    return msg, probs, res["cam"], res["info"]


def build_demo():
    import gradio as gr
    with gr.Blocks(title="Poultry Fecal-Image Screening") as demo:
        gr.Markdown(
            "# Poultry Disease Screening from Fecal Images\n"
            "Upload a close-up photo of chicken droppings. The system first checks that the "
            "image really shows poultry droppings (rejecting other animals and unrelated "
            "pictures), then screens for **Coccidiosis, Salmonellosis, Newcastle disease** "
            f"or **Healthy**.\n\n_{DISCLAIMER}_")
        with gr.Row():
            with gr.Column():
                inp = gr.Image(type="pil", label="Fecal image", sources=["upload", "webcam"])
                btn = gr.Button("Analyze", variant="primary")
            with gr.Column():
                verdict = gr.Markdown()
                probs = gr.Label(num_top_classes=4, label="Class probabilities")
                cam = gr.Image(label="Grad-CAM: regions that influenced the prediction")
                details = gr.JSON(label="Gate diagnostics")
        btn.click(run, inputs=inp, outputs=[verdict, probs, cam, details])
    return demo


demo = build_demo()

if __name__ == "__main__":
    demo.queue(max_size=16).launch(server_name="0.0.0.0",
                                   server_port=int(os.environ.get("PORT", 7860)))
'''
with open(os.path.join(CONFIG['bundle_dir'], 'poultry_core.py'), 'w') as f: f.write(CORE_SRC)
with open(os.path.join(CONFIG['bundle_dir'], 'app.py'), 'w') as f: f.write(APP_SRC)

import sys, importlib
if CONFIG['bundle_dir'] not in sys.path: sys.path.insert(0, CONFIG['bundle_dir'])
import poultry_core; importlib.reload(poultry_core)
print('[OK] poultry_core.py + app.py written to', CONFIG['bundle_dir'])

# 📦 STAGE 3: Data ingestion, manifest, quality + **leakage** audit, EDA
Exact MD5 duplicates *and* perceptual-hash (dHash) near-duplicates are located; images are grouped so that
near-identical photos can never straddle train/val/test.

In [ ]:
# ==============================================================================
# CELL 4: Google Drive mount, dataset discovery, archive & storage audit
# ==============================================================================
import os, zipfile
if SMOKE_TEST:
    CONFIG['resolved_dataset_dir'] = f'{BASE}/drive/IDSE_Dataset'
    print('[SMOKE] Skipping Google Drive.')
else:
    mp = CONFIG['drive_mount_point']
    if not os.path.ismount(mp):
        from google.colab import drive
        drive.mount(mp)
    candidates = [f'{DRIVE_ROOT}/IDSE_Dataset', f'{DRIVE_ROOT}/idse_dataset', f'{DRIVE_ROOT}/Poultry_Dataset',
                  '/content/drive/Shareddrives/IDSE_Dataset', '/content/IDSE_Dataset']
    found = next((p for p in candidates if os.path.isdir(p)), None)
    if found is None:
        for item in sorted(os.listdir(DRIVE_ROOT)):
            if any(k in item.lower() for k in ['idse', 'poultry', 'fecal', 'zenodo']) and os.path.isdir(os.path.join(DRIVE_ROOT, item)):
                found = os.path.join(DRIVE_ROOT, item); break
    if found is None:
        raise FileNotFoundError('Dataset folder not found. Put it at MyDrive/IDSE_Dataset (see README in the notebook).')
    CONFIG['resolved_dataset_dir'] = found
    print('[FOUND] dataset dir:', found)

    zips = sorted(os.path.join(r, f) for r, _, fs in os.walk(found) for f in fs if f.lower().endswith('.zip'))
    for z in zips:
        try:
            with zipfile.ZipFile(z) as zf:
                n_img = sum(1 for i in zf.infolist() if i.filename.lower().endswith(('.jpg', '.jpeg', '.png')))
            print(f'  • {os.path.relpath(z, found)}: {os.path.getsize(z)/2**20:.0f} MB, {n_img} images')
        except Exception as e:
            print(f'  • {z}: unreadable ({e})')

os.makedirs(CONFIG['experiment_output_root'], exist_ok=True)
for sub in ['checkpoints', 'plots', 'reports', 'manifests', 'splits', 'gradcam', 'final_model']:
    os.makedirs(os.path.join(CONFIG['experiment_output_root'], sub), exist_ok=True)
print('[OK] output root:', CONFIG['experiment_output_root'])

In [ ]:
# ==============================================================================
# CELL 5: Controlled extraction to local disk (or synthetic data in smoke-test mode)
# ==============================================================================
import os, shutil, zipfile
from tqdm.auto import tqdm
from PIL import Image

CLASS_MAP = {'cocci': 'Coccidiosis', 'healthy': 'Healthy', 'ncd': 'Newcastle_Disease', 'salmo': 'Salmonellosis'}
extracted_root = CONFIG['local_extracted_data']
for c in CLASS_MAP.values(): os.makedirs(os.path.join(extracted_root, c), exist_ok=True)

def _count_images():
    return sum(1 for r, _, fs in os.walk(extracted_root) for f in fs if f.lower().endswith(('.jpg', '.jpeg', '.png')))

if SMOKE_TEST and _count_images() == 0:
    rng = np.random.default_rng(0)
    base_cols = {'Coccidiosis': (150, 90, 60), 'Healthy': (90, 120, 70), 'Newcastle_Disease': (200, 190, 120), 'Salmonellosis': (70, 70, 90)}
    per_class = {'Coccidiosis': 70, 'Healthy': 70, 'Newcastle_Disease': 30, 'Salmonellosis': 70}
    for cname, n in per_class.items():
        for i in range(n):
            arr = np.clip(rng.normal(base_cols[cname], 35, (256, 256, 3)), 0, 255).astype('uint8')
            yy, xx = rng.integers(40, 216, 2); arr[yy-30:yy+30, xx-30:xx+30] = np.clip(np.array(base_cols[cname]) * rng.uniform(0.5, 1.5), 0, 255)
            Image.fromarray(arr).save(os.path.join(extracted_root, cname, f'{cname[:4].lower()}_{i:03d}.jpg'), quality=92)
        for j in range(4):   # planted exact duplicates to test the audit
            shutil.copy(os.path.join(extracted_root, cname, f'{cname[:4].lower()}_{j:03d}.jpg'), os.path.join(extracted_root, cname, f'{cname[:4].lower()}_dup{j}.jpg'))
    os.makedirs(CONFIG['non_poultry_dir'], exist_ok=True)
    for i in range(40):
        Image.fromarray(np.clip(rng.normal((30 + 5 * i % 200, 200, 230), 40, (256, 256, 3)), 0, 255).astype('uint8')).save(os.path.join(CONFIG['non_poultry_dir'], f'other_{i}.jpg'))
    print('[SMOKE] synthetic dataset generated.')
elif _count_images() > 0:
    print(f'[OK] {_count_images()} images already extracted at {extracted_root} (delete the folder to force re-extraction).')
else:
    dataset_dir = CONFIG['resolved_dataset_dir']
    zip_path = os.path.join(dataset_dir, 'zenodo_4628934_resized256.zip')
    if not os.path.isfile(zip_path):   # fallback: first zip that looks right
        cands = sorted(os.path.join(r, f) for r, _, fs in os.walk(dataset_dir) for f in fs if f.lower().endswith('.zip') and ('resized' in f.lower() or 'zenodo' in f.lower()))
        if not cands: raise FileNotFoundError('No dataset zip found.')
        zip_path = cands[0]
    print('[*] Extracting', zip_path)
    with zipfile.ZipFile(zip_path) as zf:
        members = [m for m in zf.infolist() if not m.is_dir() and m.filename.lower().endswith(('.jpg', '.jpeg', '.png'))]
        for m in tqdm(members, desc='extract'):
            parts = m.filename.replace('\\', '/').split('/')
            key = next((p.lower() for p in parts if p.lower() in CLASS_MAP), None)
            if key is None:
                fn = os.path.basename(m.filename).lower(); key = next((k for k in CLASS_MAP if fn.startswith(k)), None)
            if key:
                with zf.open(m) as src, open(os.path.join(extracted_root, CLASS_MAP[key], os.path.basename(m.filename)), 'wb') as dst:
                    shutil.copyfileobj(src, dst)

counts = {c: len([f for f in os.listdir(os.path.join(extracted_root, c)) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]) for c in CLASS_MAP.values()}
print('Extracted:', counts, '| total =', sum(counts.values()))
assert sum(counts.values()) > 0, 'No images extracted.'

In [ ]:
# ==============================================================================
# CELL 6: Manifest (MD5 + perceptual hash) + quality audit + near-duplicate groups
# ==============================================================================
import hashlib
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

def dhash64(img: Image.Image) -> int:
    g = np.asarray(img.convert('L').resize((9, 8), Image.LANCZOS), dtype=np.int16)
    bits = (g[:, 1:] > g[:, :-1]).flatten()
    return int(''.join('1' if b else '0' for b in bits), 2)

extracted_root = CONFIG['local_extracted_data']
class_to_idx = {c: i for i, c in enumerate(CONFIG['class_names'])}
files = [(c, f, os.path.join(extracted_root, c, f)) for c in CONFIG['class_names']
         for f in sorted(os.listdir(os.path.join(extracted_root, c))) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]

records = []
for cname, fname, fpath in tqdm(files, desc='manifest'):
    with open(fpath, 'rb') as fh: md5 = hashlib.md5(fh.read()).hexdigest()
    try:
        with Image.open(fpath) as im:
            w, h = im.size; mode = im.mode; fmt = im.format; im.load(); dh = dhash64(im); ok = True
    except Exception:
        w = h = 0; mode = fmt = 'CORRUPT'; dh = 0; ok = False
    records.append(dict(filename=fname, class_label=cname, class_index=class_to_idx[cname], file_path=fpath,
                        width=w, height=h, color_mode=mode, format=fmt, md5_hash=md5, dhash=np.uint64(dh), is_readable=ok))
df_manifest = pd.DataFrame(records)
n_bad = (~df_manifest.is_readable).sum()
df_manifest = df_manifest[df_manifest.is_readable].reset_index(drop=True)

print('=' * 78); print('QUALITY AUDIT'); print('=' * 78)
print(f'Images indexed        : {len(df_manifest)}  (unreadable removed: {n_bad})')
print(f'Formats / modes       : {df_manifest.format.value_counts().to_dict()} / {df_manifest.color_mode.value_counts().to_dict()}')
print(f'Dimension range       : {df_manifest.width.min()}x{df_manifest.height.min()} .. {df_manifest.width.max()}x{df_manifest.height.max()}')
dup_inst = len(df_manifest) - df_manifest.md5_hash.nunique()
conf = df_manifest.groupby('md5_hash').class_label.nunique()
print(f'Exact duplicates      : {dup_inst} redundant copies | cross-class label conflicts: {(conf > 1).sum()}')

# ---- near-duplicate clustering (single linkage on Hamming distance of 64-bit dHash) ----
_POP = np.array([bin(i).count('1') for i in range(65536)], dtype=np.uint8)
def hamming_matrix_rows(h, rows):
    x = np.bitwise_xor(h[rows][:, None], h[None, :])                # (r, N) uint64
    return _POP[x.view(np.uint16).reshape(x.shape + (4,))].sum(-1)  # popcount via 16-bit LUT

def cluster_near_duplicates(hashes, max_dist):
    n = len(hashes); parent = np.arange(n)
    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]; a = parent[a]
        return a
    for start in range(0, n, 256):
        rows = np.arange(start, min(start + 256, n))
        D = hamming_matrix_rows(hashes, rows)
        for r_i, r in enumerate(rows):
            for c in np.where(D[r_i] <= max_dist)[0]:
                if c > r:
                    ra, rb = find(r), find(c)
                    if ra != rb: parent[rb] = ra
    return np.array([find(i) for i in range(n)])

hashes = df_manifest['dhash'].values.astype(np.uint64)
max_d = CONFIG['near_dup_max_hamming']
while True:
    cl = cluster_near_duplicates(hashes, max_d)
    biggest = pd.Series(cl).value_counts().iloc[0]
    if biggest <= max(10, 0.02 * len(cl)) or max_d == 0: break
    print(f'  [guard] hamming<={max_d} merged {biggest} images into one cluster (too aggressive) -> tightening'); max_d -= 1
df_manifest['cluster_id'] = pd.factorize(cl)[0]
sizes = df_manifest.cluster_id.value_counts()
cross = df_manifest.groupby('cluster_id').class_label.nunique()
print(f'Near-duplicate groups : hamming<={max_d} -> {(sizes > 1).sum()} groups hold {int(sizes[sizes > 1].sum())} images (largest {sizes.iloc[0]})')
print(f'Groups spanning >1 class (label-noise candidates): {(cross > 1).sum()}')
CONFIG['near_dup_max_hamming_used'] = int(max_d)

df_manifest.to_csv(os.path.join(CONFIG['local_workspace'], 'manifests', 'dataset_manifest.csv'), index=False)
try: df_manifest.to_csv(os.path.join(CONFIG['experiment_output_root'], 'manifests', 'dataset_manifest.csv'), index=False)
except Exception as e: print('[warn] could not mirror manifest:', e)

In [ ]:
# ==============================================================================
# CELL 7: Exploratory data analysis
# ==============================================================================
import matplotlib.pyplot as plt
from PIL import Image
plots_dir = os.path.join(CONFIG['experiment_output_root'], 'plots')
class_counts = df_manifest.class_label.value_counts().reindex(CONFIG['class_names'])
imbalance_ratio = class_counts.max() / class_counts.min()
print(class_counts.to_string(), f'\nImbalance ratio: {imbalance_ratio:.2f}:1')

plt.figure(figsize=(9, 4.5))
bars = plt.bar(class_counts.index, class_counts.values, color=['#e74c3c', '#2ecc71', '#9b59b6', '#3498db'], edgecolor='black')
for b, c in zip(bars, class_counts.values):
    plt.text(b.get_x() + b.get_width() / 2, b.get_height() + 15, f'{c}\n({c/len(df_manifest)*100:.1f}%)', ha='center', fontweight='bold')
plt.title(f'Class distribution ({len(df_manifest):,} images)'); plt.ylim(0, class_counts.max() * 1.2); plt.grid(axis='y', ls='--', alpha=.4)
plt.tight_layout(); plt.savefig(os.path.join(plots_dir, 'eda_class_distribution.png'), dpi=150); plt.show()

fig, axes = plt.subplots(4, 4, figsize=(11, 11))
for r, cname in enumerate(CONFIG['class_names']):
    sub = df_manifest[df_manifest.class_label == cname]
    for c, (_, row) in enumerate(sub.sample(n=min(4, len(sub)), random_state=42).iterrows()):
        with Image.open(row.file_path) as im: axes[r][c].imshow(im)
        axes[r][c].axis('off'); axes[r][c].set_title(cname if c == 0 else '', fontsize=10, fontweight='bold')
plt.tight_layout(); plt.savefig(os.path.join(plots_dir, 'eda_samples.png'), dpi=130); plt.show()

# 🔀 STAGE 4: Leakage-free splitting
`split_mode`:
* **auto** (default): use the team's `friends_split_merged.csv` *only if the audit finds no val/test image with an exact or near-duplicate twin in train*; otherwise rebuild.
* **existing**: force the CSV (audit still printed).  * **rebuild**: group-aware stratified 70/15/15 split (`StratifiedGroupKFold`, groups = near-duplicate clusters).

> **Limit of any automatic audit:** photos of the *same bird / pen / day* that are not visually near-identical cannot be detected without farm/session metadata. Keep this in the limitations section of the report.

In [ ]:
# ==============================================================================
# CELL 8: Split construction, leakage audit and persistence
# ==============================================================================
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

def audit_split(df_part, name):
    m = df_part.drop(columns=['cluster_id'], errors='ignore').merge(df_manifest[['file_path', 'md5_hash', 'cluster_id']], on='file_path', how='left')
    res = {}
    for key in ['md5_hash', 'cluster_id']:
        train_keys = set(m.loc[m.split == 'train', key])
        res[key] = {s: int(m[(m.split == s)][key].isin(train_keys).sum()) for s in ['val', 'test']}
    print(f'[AUDIT:{name}] images with an exact twin in train  -> val={res["md5_hash"]["val"]}, test={res["md5_hash"]["test"]}')
    print(f'[AUDIT:{name}] images with a near-dup twin in train -> val={res["cluster_id"]["val"]}, test={res["cluster_id"]["test"]}')
    return sum(res['md5_hash'].values()) + sum(res['cluster_id'].values())

def rebuild_split():
    n_splits = 20                      # each fold ~5 % -> 3 folds test, 3 folds val, 14 folds train
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=CONFIG['random_seed'])
    fold = np.full(len(df_manifest), -1)
    for k, (_, te) in enumerate(sgkf.split(df_manifest, df_manifest.class_index, groups=df_manifest.cluster_id)): fold[te] = k
    split = np.where(fold < 3, 'test', np.where(fold < 6, 'val', 'train'))
    out = df_manifest[['filename', 'class_label', 'class_index', 'file_path', 'cluster_id']].copy(); out['split'] = split
    return out

def load_existing_split():
    p = os.path.join(CONFIG.get('resolved_dataset_dir', ''), 'splits', CONFIG['split_csv_name'])
    if not os.path.isfile(p): print(f'[info] {p} not found.'); return None
    raw = pd.read_csv(p)
    need = {'filepath', 'label', 'split'}
    if not need.issubset(raw.columns): print(f'[info] existing split lacks columns {need - set(raw.columns)}'); return None
    norm = {'cocci': 'Coccidiosis', 'healthy': 'Healthy', 'ncd': 'Newcastle_Disease', 'salmo': 'Salmonellosis',
            'coccidiosis': 'Coccidiosis', 'salmonellosis': 'Salmonellosis', 'newcastle_disease': 'Newcastle_Disease', 'salmonella': 'Salmonellosis'}
    by_name = df_manifest.set_index(['class_label', 'filename']).file_path.to_dict()
    path2cls = df_manifest.set_index('file_path').class_label.to_dict()
    recs = []
    for _, r in raw.iterrows():
        fn = os.path.basename(str(r['filepath'])); cl = norm.get(str(r['label']).lower().strip(), str(r['label']))
        fp = by_name.get((cl, fn)) or next((by_name[(c, fn)] for c in CONFIG['class_names'] if (c, fn) in by_name), None)
        if fp is None: continue
        sp = str(r['split']).lower().strip(); sp = 'val' if sp in ('val', 'validation') else ('test' if sp == 'test' else 'train')
        recs.append(dict(filename=fn, class_label=path2cls[fp], file_path=fp, split=sp))
    d = pd.DataFrame(recs); d['class_index'] = d.class_label.map(class_to_idx)
    print(f'[info] existing split resolved: {len(d)} / {len(raw)} rows matched to extracted images')
    return d

mode = CONFIG['split_mode']; chosen = None
existing = load_existing_split() if mode in ('auto', 'existing') else None
if existing is not None:
    leaks = audit_split(existing, 'existing')
    if mode == 'existing' or leaks == 0: chosen, how = existing, 'EXISTING'
    else: print(f'[DECISION] existing split has {leaks} leakage hits -> rebuilding.')
if chosen is None:
    chosen, how = rebuild_split(), 'REBUILT (group-aware)'
    assert audit_split(chosen, 'rebuilt') == 0, 'Rebuilt split still leaks!'
chosen = chosen.merge(df_manifest[['file_path', 'cluster_id']], on='file_path', how='left', suffixes=('', '_m')) if 'cluster_id' not in chosen else chosen
CONFIG['split_strategy'] = how
train_df = chosen[chosen.split == 'train'].reset_index(drop=True)
val_df = chosen[chosen.split == 'val'].reset_index(drop=True)
test_df = chosen[chosen.split == 'test'].reset_index(drop=True)
print(f'\n[SPLIT: {how}] train={len(train_df)} | val={len(val_df)} | test={len(test_df)}  (total {len(chosen)} of {len(df_manifest)} indexed)')
if len(chosen) != len(df_manifest):
    print(f'[NOTE] {len(df_manifest) - len(chosen)} indexed images are not in the split file (excluded upstream). State this in the report.')
print(pd.crosstab(chosen.split, chosen.class_label).to_string())
for n, d in [('train', train_df), ('val', val_df), ('test', test_df)]:
    d.to_csv(os.path.join(CONFIG['experiment_output_root'], 'splits', f'{n}_split.csv'), index=False)
df_all_split = chosen.copy()

In [ ]:
# ==============================================================================
# CELL 9: Class weights, architecture-specific preprocessing and the tf.data pipeline
# ==============================================================================
import pandas as pd
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import layers

cw = compute_class_weight('balanced', classes=np.arange(CONFIG['num_classes']), y=train_df.class_index.values)
CLASS_WEIGHTS = {int(i): float(round(w, 4)) for i, w in enumerate(cw)}
print('Class weights:', {CONFIG['class_names'][i]: w for i, w in CLASS_WEIGHTS.items()})

PREPROCESS_FUNCS = {a: poultry_core.get_preprocess(a) for a in CONFIG['architectures']}

# Augmentation operates on RAW 0-255 pixels (value_range matches), THEN preprocess_input is applied.
augmentation_pipeline = keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(0.08, fill_mode='reflect'),
    layers.RandomZoom(0.10, fill_mode='reflect'),
    layers.RandomTranslation(0.08, 0.08, fill_mode='reflect'),
    layers.RandomBrightness(0.10, value_range=(0, 255)),
    layers.RandomContrast(0.10, value_range=(0, 255)),
], name='poultry_augmentation')

def create_tf_dataset(df, preprocess_fn, is_training=False, batch_size=None, img_size=None):
    batch_size = batch_size or CONFIG['batch_size']; img_size = img_size or CONFIG['image_size']
    ds = tf.data.Dataset.from_tensor_slices((df['file_path'].values, df['class_index'].values.astype('int32')))
    def parse(path, y):
        img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
        img.set_shape([None, None, 3])
        img = tf.image.resize(img, img_size, method='bilinear')
        return tf.cast(tf.round(img), tf.uint8), tf.one_hot(y, CONFIG['num_classes'])   # uint8 -> cheap RAM cache
    ds = ds.map(parse, num_parallel_calls=tf.data.AUTOTUNE).cache()
    if is_training:
        ds = ds.shuffle(min(len(df), 2048), seed=CONFIG['random_seed'], reshuffle_each_iteration=True)
    ds = ds.batch(batch_size)
    def finish(x, y):
        x = tf.cast(x, tf.float32)
        if is_training:
            x = tf.clip_by_value(augmentation_pipeline(x, training=True), 0.0, 255.0)
        return preprocess_fn(x), y
    return ds.map(finish, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

xb, yb = next(iter(create_tf_dataset(train_df, PREPROCESS_FUNCS['MobileNetV2'], True)))
print(f'Pipeline OK | batch {xb.shape} | value range after MobileNetV2 preprocessing: [{float(tf.reduce_min(xb)):.2f}, {float(tf.reduce_max(xb)):.2f}]')

# 🚀 STAGE 5: Two-phase transfer learning (resumable)
Phase 1: frozen backbone + new head. Phase 2: top 30 % of backbone unfrozen (BatchNorm stays frozen) at a low learning rate.
The checkpoint can **only be improved** by phase 2 (fixes the old overwrite bug); each architecture is skipped if already trained, so a Colab disconnect never loses finished models.

In [ ]:
# ==============================================================================
# CELL 10: Model factory and training engine
# ==============================================================================
import os, gc, json, time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import tensorflow as tf, keras
from keras import layers, Model, regularizers, callbacks
from keras.applications import MobileNetV2, DenseNet169, EfficientNetB0, ResNet50

BASE_MODELS = {'MobileNetV2': MobileNetV2, 'DenseNet169': DenseNet169, 'EfficientNetB0': EfficientNetB0, 'ResNet50': ResNet50}

def build_transfer_model(name, input_shape=None):
    input_shape = input_shape or (*CONFIG['image_size'], 3)
    base = BASE_MODELS[name](weights=CONFIG['weights'], include_top=False, input_shape=input_shape)
    base.trainable = False
    inp = keras.Input(shape=input_shape, name='input_image')
    x = base(inp, training=False)
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.BatchNormalization(name='head_bn1')(x)
    x = layers.Dropout(CONFIG['dropout_rate'], name='head_dropout1')(x)
    x = layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(CONFIG['l2_regularization']), name='head_dense1')(x)
    x = layers.BatchNormalization(name='head_bn2')(x)
    x = layers.Dropout(CONFIG['dropout_rate'], name='head_dropout2')(x)
    out = layers.Dense(CONFIG['num_classes'], activation='softmax', dtype='float32', name='predictions')(x)
    model = Model(inp, out, name=f'{name}_Classifier')
    model.compile(optimizer=keras.optimizers.Adam(CONFIG['lr_head']), loss='categorical_crossentropy', metrics=['accuracy'])
    return model, base

def configure_fine_tuning(model, base):
    base.trainable = True
    n = len(base.layers); k = max(1, int(round(n * CONFIG['finetune_fraction'])))
    for l in base.layers[:n - k]: l.trainable = False
    for l in base.layers[n - k:]:
        if isinstance(l, layers.BatchNormalization): l.trainable = False     # keep BN statistics frozen
    model.compile(optimizer=keras.optimizers.Adam(CONFIG['lr_finetune']), loss='categorical_crossentropy', metrics=['accuracy'])
    print(f'   fine-tuning: top {k}/{n} backbone layers unfrozen (BN frozen)')
    return model

def plot_curves(h, name, tag=''):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4)); ep = range(1, len(h['loss']) + 1)
    ax[0].plot(ep, h['loss'], label='train'); ax[0].plot(ep, h['val_loss'], label='val'); ax[0].set_title(f'{name}{tag} - loss'); ax[0].legend(); ax[0].grid(alpha=.4)
    ax[1].plot(ep, h['accuracy'], label='train'); ax[1].plot(ep, h['val_accuracy'], label='val'); ax[1].set_title(f'{name}{tag} - accuracy'); ax[1].legend(); ax[1].grid(alpha=.4)
    plt.tight_layout(); plt.savefig(os.path.join(CONFIG['experiment_output_root'], 'plots', f'{name}{tag}_learning_curves.png'), dpi=130); plt.show()

def ckpt_path(name, tag=''): return os.path.join(CONFIG['experiment_output_root'], 'checkpoints', f'{name}{tag}_best.keras')
def summary_path(name, tag=''): return os.path.join(CONFIG['experiment_output_root'], 'reports', f'{name}{tag}_training_summary.json')

def train_model_pipeline(name, tr_df, va_df, class_weights=None, tag='', force=False):
    if os.path.isfile(summary_path(name, tag)) and os.path.isfile(ckpt_path(name, tag)) and not force:
        _s = json.load(open(summary_path(name, tag)))
        if _s.get('pipeline_version') == 2:
            print(f'[SKIP] {name}{tag} already trained with this pipeline -> loading summary'); return _s
        print(f'[RETRAIN] {name}{tag}: found a summary/checkpoint from an OLDER pipeline (buggy augmentation) -> ignoring it')
    print(f'\n{"="*78}\nTRAINING {name}{tag}\n{"="*78}')
    prep = PREPROCESS_FUNCS[name]
    ds_tr = create_tf_dataset(tr_df, prep, True); ds_va = create_tf_dataset(va_df, prep, False)
    model, base = build_transfer_model(name)
    ck = ckpt_path(name, tag)
    t0 = time.time()
    h1 = model.fit(ds_tr, validation_data=ds_va, epochs=CONFIG['epochs_head'], class_weight=class_weights, verbose=2, callbacks=[
        callbacks.EarlyStopping('val_loss', patience=CONFIG['early_stopping_patience'], restore_best_weights=True),
        callbacks.ReduceLROnPlateau('val_loss', factor=CONFIG['reduce_lr_factor'], patience=CONFIG['reduce_lr_patience'], min_lr=CONFIG['min_lr']),
        callbacks.ModelCheckpoint(ck, monitor='val_loss', save_best_only=True)])
    best_p1 = float(min(h1.history['val_loss']))
    model.load_weights(ck)                                  # start fine-tuning from the BEST phase-1 weights
    model = configure_fine_tuning(model, base)
    h2 = model.fit(ds_tr, validation_data=ds_va, epochs=CONFIG['epochs_finetune'], class_weight=class_weights, verbose=2, callbacks=[
        callbacks.EarlyStopping('val_loss', patience=5, restore_best_weights=True),
        callbacks.ReduceLROnPlateau('val_loss', factor=CONFIG['reduce_lr_factor'], patience=2, min_lr=CONFIG['min_lr']),
        callbacks.ModelCheckpoint(ck, monitor='val_loss', save_best_only=True, initial_value_threshold=best_p1)])  # never overwrite a better phase-1 model
    secs = time.time() - t0
    hist = {k: h1.history[k] + h2.history.get(k, []) for k in ['loss', 'val_loss', 'accuracy', 'val_accuracy']}
    plot_curves(hist, name, tag)
    summ = dict(pipeline_version=2, model_name=name, tag=tag, total_params=int(model.count_params()), checkpoint_path=ck,
                model_size_mb=round(os.path.getsize(ck) / 2**20, 2), training_time_sec=round(secs, 1),
                best_val_loss=round(float(min(hist['val_loss'])), 4), best_val_accuracy=round(float(max(hist['val_accuracy'])), 4),
                history={k: [round(float(v), 5) for v in vs] for k, vs in hist.items()})
    json.dump(summ, open(summary_path(name, tag), 'w'), indent=2)
    del model, base, ds_tr, ds_va; keras.backend.clear_session(); gc.collect()
    return summ
print('[OK] training engine ready')

In [ ]:
# ==============================================================================
# CELL 11: Train all architectures (resumable - already finished models are skipped)
# ==============================================================================
TRAIN_SUMMARIES = {}
for arch in CONFIG['architectures']:
    TRAIN_SUMMARIES[arch] = train_model_pipeline(arch, train_df, val_df, class_weights=CLASS_WEIGHTS)
    s = TRAIN_SUMMARIES[arch]
    print(f"-> {arch}: best val loss {s['best_val_loss']} | best val acc {s['best_val_accuracy']*100:.2f}% | {s.get('training_time_sec', 0):.0f}s")

# 📊 STAGE 6: Benchmark on validation → pick one model → evaluate **once** on the test set
Selection metric = validation **macro-F1**. Bootstrap 95 % confidence intervals are reported so differences of <1 % are not over-interpreted.

In [ ]:
# ==============================================================================
# CELL 12: Evaluation engine
# ==============================================================================
import time, gc
import numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report, roc_auc_score, log_loss)
from sklearn.preprocessing import label_binarize

def evaluate_model_comprehensive(name, split_df, split_name='val', tag=''):
    ck = ckpt_path(name, tag)
    if not os.path.isfile(ck): raise FileNotFoundError(ck)
    model = keras.models.load_model(ck, compile=False)
    ds = create_tf_dataset(split_df, PREPROCESS_FUNCS[name], False)
    for xw, _ in ds.take(1): model.predict(xw, verbose=0)           # warm-up
    t0 = time.time(); y_prob = model.predict(ds, verbose=0); ms = (time.time() - t0) / len(split_df) * 1000
    y_pred = y_prob.argmax(1); y_true = split_df.class_index.values; C = CONFIG['num_classes']
    yb = label_binarize(y_true, classes=range(C))
    try: auc_m = roc_auc_score(yb, y_prob, average='macro')
    except Exception: auc_m = float('nan')
    m = {'Model': name + tag, 'Split': split_name, 'Accuracy': accuracy_score(y_true, y_pred), 'Balanced_Acc': balanced_accuracy_score(y_true, y_pred),
         'Macro_Precision': precision_score(y_true, y_pred, average='macro', zero_division=0), 'Macro_Recall': recall_score(y_true, y_pred, average='macro', zero_division=0),
         'Macro_F1': f1_score(y_true, y_pred, average='macro', zero_division=0), 'Weighted_F1': f1_score(y_true, y_pred, average='weighted', zero_division=0),
         'ROC_AUC_Macro': auc_m, 'Log_Loss': log_loss(y_true, y_prob, labels=list(range(C))), 'Latency_ms': ms,
         'Params_M': model.count_params() / 1e6, 'Size_MB': os.path.getsize(ck) / 2**20}
    for i, c in enumerate(CONFIG['class_names']):
        m[f'Recall_{c}'] = recall_score(y_true, y_pred, labels=[i], average=None, zero_division=0)[0]
        m[f'F1_{c}'] = f1_score(y_true, y_pred, labels=[i], average=None, zero_division=0)[0]
    m = {k: (round(float(v), 4) if isinstance(v, (float, np.floating)) else v) for k, v in m.items()}
    report = classification_report(y_true, y_pred, labels=list(range(C)), target_names=CONFIG['class_names'], digits=4, zero_division=0)
    del model, ds; keras.backend.clear_session(); gc.collect()
    return m, confusion_matrix(y_true, y_pred, labels=list(range(C))), report, y_true, y_pred, y_prob

def bootstrap_ci(y_true, y_pred, n=1000, seed=42):
    rng = np.random.default_rng(seed); N = len(y_true); acc, f1 = [], []
    for _ in range(n):
        i = rng.integers(0, N, N); acc.append((y_true[i] == y_pred[i]).mean()); f1.append(f1_score(y_true[i], y_pred[i], average='macro', zero_division=0))
    return (np.percentile(acc, [2.5, 97.5]), np.percentile(f1, [2.5, 97.5]))
print('[OK] evaluation engine ready')

In [ ]:
# ==============================================================================
# CELL 13: Validation benchmark, model selection, FINAL test evaluation (once)
# ==============================================================================
VAL_BENCH = []
for arch in CONFIG['architectures']:
    m, *_ = evaluate_model_comprehensive(arch, val_df, 'val'); VAL_BENCH.append(m)
df_comparison = pd.DataFrame(VAL_BENCH).sort_values(['Macro_F1', 'Latency_ms'], ascending=[False, True]).reset_index(drop=True)
print('=' * 78); print('VALIDATION BENCHMARK'); print('=' * 78)
print(df_comparison[['Model', 'Macro_F1', 'Balanced_Acc', 'Accuracy', 'Recall_Newcastle_Disease', 'ROC_AUC_Macro', 'Latency_ms', 'Params_M', 'Size_MB']].to_string(index=False))
df_comparison.to_csv(os.path.join(CONFIG['experiment_output_root'], 'reports', 'model_comparison_val.csv'), index=False)

SELECTED = df_comparison.iloc[0]['Model']; CONFIG['selected_baseline'] = SELECTED
print(f'\n>>> SELECTED MODEL (by validation macro-F1): {SELECTED}')

test_metrics, test_cm, test_report, test_yt, test_yp, test_yprob = evaluate_model_comprehensive(SELECTED, test_df, 'test')
acc_ci, f1_ci = bootstrap_ci(test_yt, test_yp)
print('\n' + '=' * 78); print(f'FINAL TEST RESULT ({SELECTED}, n={len(test_df)})'); print('=' * 78)
print(f"Accuracy    : {test_metrics['Accuracy']*100:.2f}%  (95% CI {acc_ci[0]*100:.2f}-{acc_ci[1]*100:.2f})")
print(f"Macro-F1    : {test_metrics['Macro_F1']:.4f}  (95% CI {f1_ci[0]:.4f}-{f1_ci[1]:.4f})")
print(f"Balanced acc: {test_metrics['Balanced_Acc']*100:.2f}% | NCD recall: {test_metrics['Recall_Newcastle_Disease']*100:.1f}%")
print('\n' + test_report)
plt.figure(figsize=(6.5, 5.5)); sns.heatmap(test_cm, annot=True, fmt='d', cmap='Blues', xticklabels=CONFIG['class_names'], yticklabels=CONFIG['class_names'])
plt.title(f'{SELECTED} - test confusion matrix'); plt.ylabel('True'); plt.xlabel('Predicted'); plt.tight_layout()
plt.savefig(os.path.join(CONFIG['experiment_output_root'], 'plots', f'{SELECTED}_test_confusion_matrix.png'), dpi=150); plt.show()

test_res_df = test_df.copy(); test_res_df['predicted_class'] = [CONFIG['class_names'][i] for i in test_yp]
for i, c in enumerate(CONFIG['class_names']): test_res_df[f'prob_{c}'] = test_yprob[:, i]
test_res_df.to_csv(os.path.join(CONFIG['experiment_output_root'], 'reports', f'{SELECTED}_test_predictions.csv'), index=False)
TEST_CI = dict(acc=acc_ci.tolist(), f1=f1_ci.tolist())

# 🔎 STAGE 7 (optional): Why are our numbers lower than the papers? — *paper-style (leaky) protocol*
Many of the cited studies (Papers 2, 3, 8, 12 …) **balance the data by augmenting / oversampling the whole dataset first and split afterwards**. Augmented copies of one photo then sit in both train and test, which inflates accuracy.
This cell re-creates that protocol (random oversampling by duplication → random 80/10/10 split) and trains the *selected* model on it.
**This number is diagnostic only. Never report it as model performance.** Enable with `CONFIG['run_paper_style_comparison'] = True`.

In [ ]:
# ==============================================================================
# CELL 14: Paper-style (leaky) protocol vs our leakage-free protocol  [optional, diagnostic]
# ==============================================================================
from sklearn.model_selection import train_test_split
PAPER_STYLE = None
if CONFIG['run_paper_style_comparison']:
    mx = df_manifest.class_label.value_counts().max()
    big = pd.concat([g.sample(n=mx, replace=True, random_state=42) for _, g in df_manifest.groupby('class_label')]).sample(frac=1, random_state=42).reset_index(drop=True)
    big = big[['filename', 'class_label', 'class_index', 'file_path']]
    ps_tr, tmp = train_test_split(big, test_size=0.2, stratify=big.class_index, random_state=42)
    ps_va, ps_te = train_test_split(tmp, test_size=0.5, stratify=tmp.class_index, random_state=42)
    ps_tr, ps_va, ps_te = ps_tr.reset_index(drop=True), ps_va.reset_index(drop=True), ps_te.reset_index(drop=True)
    twin = ps_te.file_path.isin(set(ps_tr.file_path)).mean()
    print(f'Paper-style split: train={len(ps_tr)} val={len(ps_va)} test={len(ps_te)} | {twin*100:.1f}% of its TEST images are byte-identical to a TRAIN image')
    train_model_pipeline(SELECTED, ps_tr, ps_va, class_weights=None, tag='_paperstyle')
    ps_m, *_ = evaluate_model_comprehensive(SELECTED, ps_te, 'test', tag='_paperstyle')
    PAPER_STYLE = dict(test_accuracy=ps_m['Accuracy'], macro_f1=ps_m['Macro_F1'], test_twin_fraction=float(twin))
    print('\n' + '=' * 78)
    print(f"{SELECTED} | leakage-free protocol : acc {test_metrics['Accuracy']*100:.2f}%  macroF1 {test_metrics['Macro_F1']:.4f}")
    print(f"{SELECTED} | paper-style (leaky)   : acc {ps_m['Accuracy']*100:.2f}%  macroF1 {ps_m['Macro_F1']:.4f}   <- inflated by duplicates, NOT a valid result")
    print(f"Gap attributable to the evaluation protocol: {(ps_m['Accuracy'] - test_metrics['Accuracy'])*100:+.2f} percentage points")
    print('=' * 78)
else:
    print('[skipped] set CONFIG["run_paper_style_comparison"]=True to quantify the protocol-induced gap.')

# 🔬 STAGE 8: Explainability, calibration, robustness, error analysis

In [ ]:
# ==============================================================================
# CELL 15: Grad-CAM (working), reliability diagram / ECE, robustness (incl. JPEG), error analysis
# ==============================================================================
import io
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score

best_model = keras.models.load_model(ckpt_path(SELECTED), compile=False)
prep_fn = PREPROCESS_FUNCS[SELECTED]

# ---- Grad-CAM: one correct example per class (shared implementation with the web app) -------------
fig, axes = plt.subplots(2, CONFIG['num_classes'], figsize=(3.2 * CONFIG['num_classes'], 6.6))
for j, cname in enumerate(CONFIG['class_names']):
    sub = test_res_df[(test_res_df.class_label == cname)]
    row = (sub[sub.predicted_class == cname].iloc[0] if (sub.predicted_class == cname).any() else sub.iloc[0])
    arr = poultry_core.to_model_array(row.file_path, CONFIG['image_size'])
    x = prep_fn(arr.copy())[None].astype('float32')
    cam, cls = poultry_core.gradcam_heatmap(best_model, x)
    axes[0][j].imshow(Image.open(row.file_path).convert('RGB')); axes[0][j].set_title(f'True: {cname}', fontsize=9); axes[0][j].axis('off')
    axes[1][j].imshow(poultry_core.overlay_cam(Image.open(row.file_path), cam)); axes[1][j].set_title(f'Pred: {CONFIG["class_names"][cls]}', fontsize=9); axes[1][j].axis('off')
plt.suptitle(f'Grad-CAM - {SELECTED}'); plt.tight_layout()
plt.savefig(os.path.join(CONFIG['experiment_output_root'], 'plots', f'{SELECTED}_gradcam.png'), dpi=140); plt.show()

# ---- confidence rejection + reliability diagram ---------------------------------------------------
conf = test_yprob.max(1); correct = (test_yp == test_yt); thr = CONFIG['confidence_threshold']; acc_mask = conf >= thr
print(f'Raw accuracy {correct.mean()*100:.2f}% | threshold {thr}: coverage {acc_mask.mean()*100:.1f}% , accuracy on accepted {correct[acc_mask].mean()*100:.2f}%')
bins = np.linspace(0, 1, 11); ece = 0.0; bx, by = [], []
for lo, hi in zip(bins[:-1], bins[1:]):
    msk = (conf > lo) & (conf <= hi)
    if msk.any(): ece += abs(conf[msk].mean() - correct[msk].mean()) * msk.mean(); bx.append(conf[msk].mean()); by.append(correct[msk].mean())
plt.figure(figsize=(4.5, 4.5)); plt.plot([0, 1], [0, 1], 'k--'); plt.plot(bx, by, 'o-'); plt.xlabel('confidence'); plt.ylabel('accuracy'); plt.title(f'Reliability (ECE={ece:.4f})'); plt.grid(alpha=.4)
plt.tight_layout(); plt.savefig(os.path.join(CONFIG['experiment_output_root'], 'plots', 'reliability.png'), dpi=140); plt.show()

# ---- robustness (all four perturbations the markdown promised) ------------------------------------
def perturb(path, kind):
    im = Image.open(path).convert('RGB'); a = np.asarray(im)
    if kind == 'blur':
        import cv2; a = cv2.GaussianBlur(a, (9, 9), 0)
    elif kind == 'brightness_down': a = np.clip(a.astype(np.float32) * 0.7 - 20, 0, 255).astype(np.uint8)
    elif kind == 'gaussian_noise': a = np.clip(a.astype(np.int16) + np.random.default_rng(0).normal(0, 15, a.shape), 0, 255).astype(np.uint8)
    elif kind == 'jpeg_q20':
        buf = io.BytesIO(); im.save(buf, 'JPEG', quality=20); a = np.asarray(Image.open(buf).convert('RGB'))
    return poultry_core.to_model_array(Image.fromarray(a), CONFIG['image_size'])
sub = test_df.sample(n=min(300, len(test_df)), random_state=42); rob = []
for kind in ['clean', 'blur', 'brightness_down', 'gaussian_noise', 'jpeg_q20']:
    X = np.stack([prep_fn((poultry_core.to_model_array(p, CONFIG['image_size']) if kind == 'clean' else perturb(p, kind)).copy()) for p in sub.file_path]).astype('float32')
    pr = best_model.predict(X, batch_size=32, verbose=0).argmax(1)
    rob.append(dict(Condition=kind, Accuracy=round(accuracy_score(sub.class_index, pr), 4), Macro_F1=round(f1_score(sub.class_index, pr, average='macro'), 4)))
df_rob = pd.DataFrame(rob); print('\nRobustness:\n', df_rob.to_string(index=False)); df_rob.to_csv(os.path.join(CONFIG['experiment_output_root'], 'reports', 'robustness.csv'), index=False)

# ---- error analysis -------------------------------------------------------------------------------
err = test_res_df[test_res_df.class_label != test_res_df.predicted_class]
print(f'\nErrors: {len(err)}/{len(test_res_df)} ({len(err)/len(test_res_df)*100:.2f}%)')
print(err.groupby(['class_label', 'predicted_class']).size().sort_values(ascending=False).head(6).to_string())
del best_model; keras.backend.clear_session()

# 🆕 STAGE 9 — NOVELTY: *Is this actually poultry droppings?*  (poultry vs. other-animal rejection)

**Can it be done with no dataset?** — *Building* it: **yes**. *Proving it works*: needs a small evaluation set.

| Layer | Needs extra training data? | What it does | Weak spot |
|---|---|---|---|
| **A. CLIP zero-shot** (pretrained, 24 text prompts: chicken droppings vs cow dung, dog feces, human stool, … , live chicken, person, landscape …) | **No** | Decides "poultry droppings vs other animal / not feces" | Chicken vs. *turkey/duck/wild-bird* droppings is hard for CLIP; probabilities are not calibrated |
| **B. Mahalanobis feature-space detector** on the classifier's own features | **No** (fitted on the poultry training set only) | "Does this look like the poultry photos I was trained on?" | May react to *background / camera* differences as much as to species |
| **C. Softmax confidence** | No | Rejects ambiguous predictions | Over-confident on unfamiliar inputs → that is why A and B exist |

Thresholds are calibrated **only on genuine poultry validation images**, so that each gate keeps ≥ `gate_accept_rate` (97.5 %) of them.
The *rejection rate on other animals* is then an **evaluation** quantity: put ~50–200 non-poultry images (cow dung, dog/cat feces, human stool, horse/goat/sheep droppings, plus a few non-feces pictures; check each source's licence) into
`MyDrive/non_poultry_eval/<animal>/*.jpg`. They are **never used for training or threshold-setting**, only to measure the gate.
Without that folder the notebook still builds and deploys the gate, but reports its non-poultry rejection rate as **unmeasured** (no number is invented).

In [ ]:
# ==============================================================================
# CELL 16: Fit & calibrate the gates (poultry data only - no extra dataset)
# ==============================================================================
import poultry_core as pc
best_model = keras.models.load_model(ckpt_path(SELECTED), compile=False)
feat_model = keras.Model(best_model.inputs, best_model.get_layer('gap').output)
prep_sel = PREPROCESS_FUNCS[SELECTED]
extract = lambda df: feat_model.predict(create_tf_dataset(df, prep_sel, False), verbose=0)
F_tr, F_va, F_te = extract(train_df), extract(val_df), extract(test_df)
print('feature dims:', F_tr.shape)

ACCEPT = CONFIG['gate_accept_rate']
maha_gate = pc.MahalanobisGate.fit(F_tr, train_df.class_index.values, CONFIG['num_classes'])
s_tr, s_va, s_te = maha_gate.score(F_tr), maha_gate.score(F_va), maha_gate.score(F_te)
MAHA_THR = pc.threshold_accept_low(s_va, ACCEPT)
print(f'[Gate B] feature-outlier score  train median {np.median(s_tr):.3f} | val median {np.median(s_va):.3f} | threshold {MAHA_THR:.3f}')
print(f'         genuine poultry kept: val {(s_va <= MAHA_THR).mean()*100:.1f}% | test {(s_te <= MAHA_THR).mean()*100:.1f}%')
maha_gate.save(os.path.join(CONFIG['local_workspace'], 'maha.npz'))

CLIP_CFG = dict(enabled=False, model_name=CONFIG['clip_model'])
CLIP_THR = None
if CONFIG['enable_clip']:
    try:
        clip_gate = pc.ClipGate(CONFIG['clip_model'])
        p_va, top_va = clip_gate.score(val_df.file_path.tolist())
        CLIP_THR = pc.threshold_accept_high(p_va, ACCEPT)
        print(f'[Gate A] CLIP p(poultry) on genuine val images: median {np.median(p_va):.3f}, 5th pct {np.percentile(p_va, 5):.3f} | threshold {CLIP_THR:.4f}')
        top_labels = pd.Series(top_va).value_counts().head(4).to_dict(); print('         most common CLIP best-matches on real poultry images:', top_labels)
        if CLIP_THR < 0.2: print('         [WARNING] threshold is low -> CLIP separates this image style poorly; Gate B does most of the work.')
        CLIP_CFG.update(enabled=True, threshold=CLIP_THR, accept_rate=ACCEPT)
        del clip_gate
    except Exception as e:
        print(f'[Gate A] CLIP unavailable here ({type(e).__name__}: {e}) -> deploying without it.')
else:
    print('[Gate A] CLIP disabled in CONFIG.')
del best_model, feat_model; keras.backend.clear_session()

# 📦 STAGE 10: Deployment bundle (Hugging Face Space / Docker / local) and quality assurance

In [ ]:
# ==============================================================================
# CELL 17: Export the deployment bundle
# ==============================================================================
import shutil, zipfile
BUNDLE = CONFIG['bundle_dir']; MB = os.path.join(BUNDLE, 'model_bundle'); os.makedirs(MB, exist_ok=True)
shutil.copy(ckpt_path(SELECTED), os.path.join(MB, 'model.keras'))
shutil.copy(os.path.join(CONFIG['local_workspace'], 'maha.npz'), os.path.join(MB, 'maha.npz'))

meta = dict(arch=SELECTED, class_names=CONFIG['class_names'], image_size=list(CONFIG['image_size']),
            conf_threshold=CONFIG['confidence_threshold'], feature_layer='gap',
            maha=dict(threshold=float(MAHA_THR), accept_rate=ACCEPT, fitted_on='train', calibrated_on='val'),
            clip=CLIP_CFG,
            evaluation=dict(split_strategy=CONFIG['split_strategy'], n_test=len(test_df),
                            test_accuracy=test_metrics['Accuracy'], test_macro_f1=test_metrics['Macro_F1'],
                            test_accuracy_ci95=TEST_CI['acc'], test_macro_f1_ci95=TEST_CI['f1']),
            versions=dict(tensorflow=tf.__version__, keras=keras.__version__), created=CONFIG['timestamp'])
json.dump(meta, open(os.path.join(MB, 'meta.json'), 'w'), indent=2)

import importlib, subprocess, sys
try: import gradio
except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'gradio'])
import gradio
reqs = [f'tensorflow-cpu=={tf.__version__}', f'keras=={keras.__version__}', 'numpy', 'pillow', f'gradio=={gradio.__version__}']
if CLIP_CFG['enabled']:
    reqs += ['--extra-index-url https://download.pytorch.org/whl/cpu', 'torch', 'transformers>=4.40']
open(os.path.join(BUNDLE, 'requirements.txt'), 'w').write('\n'.join(reqs) + '\n')

open(os.path.join(BUNDLE, 'README.md'), 'w').write(f"""---
title: Poultry Fecal-Image Screening
emoji: 🐔
colorFrom: yellow
colorTo: green
sdk: gradio
sdk_version: {gradio.__version__}
python_version: "{sys.version_info.major}.{sys.version_info.minor}"
app_file: app.py
pinned: false
---
# Poultry Disease Screening from Fecal Images
Four-class screening (Coccidiosis / Healthy / Newcastle disease / Salmonellosis) with a poultry-vs-other-animal rejection gate.
Backbone: **{SELECTED}** - test accuracy {test_metrics['Accuracy']*100:.2f}% (leakage-free split, n={len(test_df)}).
Research prototype - not a substitute for laboratory diagnosis.

## Deploy
* **Hugging Face Spaces (free CPU):** create a *Gradio* Space and upload this whole folder (web UI "Upload files" handles the large model file via LFS automatically), or run the final notebook cell with `CONFIG['push_to_hf_space']` set.
* **Docker (Render / Railway / Cloud Run / any VM):** `docker build -t poultry . && docker run -p 7860:7860 poultry`
* **Local:** `pip install -r requirements.txt && python app.py` -> http://localhost:7860
* Set `USE_CLIP=0` to run without the CLIP gate (faster start, ~1 GB less RAM).
""")
open(os.path.join(BUNDLE, 'Dockerfile'), 'w').write(f"""FROM python:{sys.version_info.major}.{sys.version_info.minor}-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
ENV PORT=7860
EXPOSE 7860
CMD ["python", "app.py"]
""")
open(os.path.join(BUNDLE, '.dockerignore'), 'w').write('__pycache__\n*.pyc\n')

try:
    model_for_export = keras.models.load_model(ckpt_path(SELECTED), compile=False)
    import contextlib
    with open(os.devnull, 'w') as _dn, contextlib.redirect_stdout(_dn), contextlib.redirect_stderr(_dn):
        model_for_export.export(os.path.join(CONFIG['experiment_output_root'], 'final_model', SELECTED + '_savedmodel'))   # TF-Serving / TFLite route (old code used .save(dir) which crashes in Keras 3)
    print('[SAVED] SavedModel export (TF-Serving / TFLite route)')
    del model_for_export
except Exception as e: print('[info] SavedModel export skipped:', e)

zip_path = os.path.join(CONFIG['experiment_output_root'], 'poultry_deploy_app.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for r, _, fs in os.walk(BUNDLE):
        for f in fs:
            if '__pycache__' not in r: zf.write(os.path.join(r, f), os.path.relpath(os.path.join(r, f), BUNDLE))
print('Bundle contents:'); [print('  ', os.path.relpath(os.path.join(r, f), BUNDLE), f'{os.path.getsize(os.path.join(r, f))/2**20:.1f} MB') for r, _, fs in os.walk(BUNDLE) for f in fs if '__pycache__' not in r]
print('[SAVED] deployable zip:', zip_path)

In [ ]:
# ==============================================================================
# CELL 18: QA on the DEPLOYED code path + gate evaluation
# ==============================================================================
from sklearn.metrics import roc_auc_score
analyzer = pc.PoultryAnalyzer(BUNDLE + '/model_bundle')
print('Analyzer loaded | arch:', analyzer.arch, '| CLIP gate active:', analyzer.clip is not None, '| warnings:', analyzer.warnings)

# 1) train/serve consistency: deployed preprocessing must reproduce the notebook's test probabilities
n_chk = min(60, len(test_df)); chk = analyzer.score_batch(test_df.file_path.tolist()[:n_chk])
max_diff = np.abs(chk['probs'] - test_yprob[:n_chk]).max(); agree = (chk['probs'].argmax(1) == test_yp[:n_chk]).mean()
print(f'[QA-1] max |p_deployed - p_notebook| = {max_diff:.4f} | argmax agreement = {agree*100:.1f}%  ->', 'OK' if (max_diff < 0.05 and agree >= 0.98) else '!!! PREPROCESSING MISMATCH - DO NOT DEPLOY !!!')

# 2) end-to-end single image
res = analyzer.analyze(test_df.file_path.iloc[0]); print('[QA-2] example:', res['verdict'], res['info']['predicted_class'], res['info']['confidence'], '| Grad-CAM produced:', res['cam'] is not None)

# 3) gate behaviour on genuine poultry test images (all of these SHOULD be accepted)
pos = analyzer.score_batch(test_df.file_path.tolist())
pos_verdicts = [analyzer.decide(pos['probs'][i], None if pos['maha'] is None else pos['maha'][i], None if pos['clip_p'] is None else pos['clip_p'][i])[0] for i in range(len(test_df))]
pos_gate_pass = np.mean([v in (pc.VERDICT_PREDICTION, pc.VERDICT_UNCERTAIN) for v in pos_verdicts])
print(f'[QA-3] genuine poultry TEST images passing both gates: {pos_gate_pass*100:.1f}% (target ~ {ACCEPT**2*100:.0f}-{ACCEPT*100:.0f}%)')

# 4) non-poultry evaluation (optional folder; never used for training/thresholds)
GATE_EVAL = dict(poultry_pass_rate=float(pos_gate_pass), measured=False)
neg_dir = CONFIG['non_poultry_dir']
neg_paths = [os.path.join(r, f) for r, _, fs in os.walk(neg_dir) for f in fs if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))] if os.path.isdir(neg_dir) else []
if neg_paths:
    neg = analyzer.score_batch(neg_paths)
    neg_verdicts = [analyzer.decide(neg['probs'][i], None if neg['maha'] is None else neg['maha'][i], None if neg['clip_p'] is None else neg['clip_p'][i])[0] for i in range(len(neg_paths))]
    rej = np.array([v in (pc.VERDICT_NOT_POULTRY, pc.VERDICT_OOD) for v in neg_verdicts])
    no_gate_false_alarm = (neg['probs'].max(1) >= analyzer.conf_thr).mean()
    print(f'\n[QA-4] {len(neg_paths)} non-poultry images from {neg_dir}')
    print(f'   combined gate: rejected {rej.mean()*100:.1f}% as non-poultry/unfamiliar')
    print(f'   WITHOUT any gate, {no_gate_false_alarm*100:.1f}% would have received a confident (>= {analyzer.conf_thr}) poultry-disease label')
    for gate, ps, ns, sign in [('B Mahalanobis', pos['maha'], neg['maha'], -1), ('A CLIP', pos['clip_p'], neg['clip_p'], +1)]:
        if ps is None: continue
        auc = roc_auc_score(np.r_[np.ones(len(ps)), np.zeros(len(ns))], sign * np.r_[ps, ns])
        print(f'   AUROC (poultry vs non-poultry) gate {gate}: {auc:.3f}')
    by = pd.DataFrame({'folder': [os.path.basename(os.path.dirname(p)) for p in neg_paths], 'rejected': rej}).groupby('folder').rejected.agg(['mean', 'count'])
    print('   rejection rate per folder:\n', by.rename(columns={'mean': 'rejected_rate'}).round(3).to_string())
    GATE_EVAL.update(measured=True, n_negatives=len(neg_paths), rejection_rate=float(rej.mean()), no_gate_false_alarm=float(no_gate_false_alarm))
    print('   CAVEAT: if the negatives differ from poultry photos in background/camera, a high score may reflect that, not species recognition.')
else:
    print(f'\n[QA-4] No folder at {neg_dir}. Gate deployed but its non-poultry rejection rate is UNMEASURED. Add images and re-run this cell to quantify it.')

In [ ]:
# ==============================================================================
# CELL 19: Optional - Hugging Face Space push and a temporary public demo link
# ==============================================================================
if CONFIG['push_to_hf_space']:
    from huggingface_hub import HfApi
    try:
        from google.colab import userdata; token = userdata.get('HF_TOKEN')
    except Exception: token = os.environ.get('HF_TOKEN')
    api = HfApi(token=token)
    api.create_repo(repo_id=CONFIG['push_to_hf_space'], repo_type='space', space_sdk='gradio', exist_ok=True)
    api.upload_folder(folder_path=BUNDLE, repo_id=CONFIG['push_to_hf_space'], repo_type='space', ignore_patterns=['__pycache__/*'])
    print('[DEPLOYED] https://huggingface.co/spaces/' + CONFIG['push_to_hf_space'], '(first build takes several minutes)')
else:
    print('[info] CONFIG["push_to_hf_space"] is None -> upload poultry_deploy_app.zip contents manually (see README in the bundle).')

if CONFIG['launch_demo']:
    sys.path.insert(0, BUNDLE); os.environ.setdefault('PORT', '7860')
    import app as poultry_app
    poultry_app.demo.queue(max_size=8).launch(share=True, debug=False)   # temporary public link for final verification

# 📋 STAGE 11: Auto-generated project report (every number is computed, none hard-coded)

In [ ]:
# ==============================================================================
# CELL 20: Final report (with a like-for-like comparison against the 14 cited papers)
# ==============================================================================
paper_rows = [
 ('1 Machuve 2022',        '6,812 farm + 1,255 PCR-labelled', '98.24% (Xception, FT + frozen BN)',   'Test set = farm-labelled images (expert annotation, not PCR). Original dataset paper.'),
 ('2 Degu 2023',           '8,067 -> 10,500 (ROS + aug)',      '98.70% (ResNet50 on YOLOv3 ROI)',    'Oversampling/augmentation before training; order vs. split not stated in our summary - verify in paper.'),
 ('3 Islam 2025',          '8,067 -> 12,000 (aug)',            '97.04% (RF-selected features + ANN)','5-fold CV; same augmentation-order question.'),
 ('4 Alam 2026',           '4,090 (different, smaller set)',   '98% (tuned DenseNet169)',            'Different dataset - not directly comparable.'),
 ('5 Luo 2025',            '6,812 (aug to ~2,100/class)',      '98.3% (ResNet152 + SVM)',            'Frozen CNN features + classical classifier.'),
 ('6 Shanmugapriya 2026',  '6,812 + 14,618 (Nigeria)',         '92.0% (SPFRNN)',                     'Two-level system + IoT simulation; different task.'),
 ('7 Dhungana 2025',       '6,812 (70/20/10)',                 '99.12% (YOLO11 crop + EfficientNet-B0)', 'Classifies YOLO-cropped regions; the report notes no field test.'),
 ('8 Anwarul 2025',        '8,067 -> 10,500',                  '98.4% (DeepFowl)',                   'Oversampling algorithm; 120x120 input.'),
 ('9 Lee 2026',            '8,420 (balanced ~2,100/class)',    '98.89% (Fecal-DANet)',               'Class counts imply augmentation (Zenodo NCD has only ~562 originals) - verify split order.'),
 ('10 Tasdelen 2025',      '6,812 -> 9,068',                   '97.1% (MobileNetV2, 96x96)',         '5-fold CV; frozen backbone.'),
 ('11 Hossain 2023',       '1,080 (270/class)',                '92.8% on heterogeneous farm test',   'Drops from 99% validation to 92.8% on new farms/phones: the realistic figure.'),
 ('12 Ahmed 2025',         '14,879 -> 20,000',                 '99.67% (YOLOv8)',                    'Merges two datasets that may overlap (8,067 = 1,255 + 6,812); the P/R/Acc/F1 columns in the report rise in equal 0.0011 steps - treat with caution.'),
 ('13 Al Huda 2024',       '6,812 (5448/682/682)',             '97.78% (EffNetV2B0 + DenseNet121)',  'Closest protocol to ours.'),
 ('14 Harshitha 2024',     'microscopy, binary',               '96.5% (VGG)',                        'Different modality (microscope images) - not comparable.'),
]
df_papers = pd.DataFrame(paper_rows, columns=['Paper', 'Data', 'Best reported', 'Comparability note'])
try: papers_md = df_papers.to_markdown(index=False); bench_md = df_comparison[['Model', 'Macro_F1', 'Balanced_Acc', 'Accuracy', 'Recall_Newcastle_Disease', 'Latency_ms', 'Params_M', 'Size_MB']].to_markdown(index=False)
except Exception: papers_md = df_papers.to_string(index=False); bench_md = df_comparison[['Model', 'Macro_F1', 'Accuracy']].to_string(index=False)

ps_txt = (f"Paper-style (leaky) protocol on the same model: {PAPER_STYLE['test_accuracy']*100:.2f}% accuracy, with {PAPER_STYLE['test_twin_fraction']*100:.1f}% of test images duplicated in train -> "
          f"inflation of {(PAPER_STYLE['test_accuracy']-test_metrics['Accuracy'])*100:+.2f} points. Diagnostic only." if PAPER_STYLE else 'Paper-style protocol comparison was not run.')
gate_txt = (f"Non-poultry evaluation ({GATE_EVAL['n_negatives']} images): {GATE_EVAL['rejection_rate']*100:.1f}% rejected by the combined gate; without the gate {GATE_EVAL['no_gate_false_alarm']*100:.1f}% would have received a confident disease label. "
            f"Genuine poultry test images passing the gate: {GATE_EVAL['poultry_pass_rate']*100:.1f}%." if GATE_EVAL['measured'] else
            f"Non-poultry rejection rate NOT measured (no evaluation folder). Genuine poultry test images passing the gate: {GATE_EVAL['poultry_pass_rate']*100:.1f}%.")

report = f"""# Final Report - Poultry Disease Classification from Fecal Images
Run `{CONFIG['timestamp']}` | selected model **{SELECTED}** | split: {CONFIG['split_strategy']}

## 1. Data and protocol
{len(df_manifest):,} readable images indexed; split sizes train/val/test = {len(train_df)}/{len(val_df)}/{len(test_df)}.
Group-aware / leakage-audited (exact MD5 and perceptual-hash near-duplicates). Class weights (not oversampling) handle imbalance. Augmentation is applied to training batches only.

## 2. Validation benchmark
{bench_md}

## 3. Held-out test result ({SELECTED}, evaluated once)
* Accuracy **{test_metrics['Accuracy']*100:.2f}%** (95% CI {TEST_CI['acc'][0]*100:.2f}-{TEST_CI['acc'][1]*100:.2f})
* Macro-F1 **{test_metrics['Macro_F1']:.4f}** (95% CI {TEST_CI['f1'][0]:.4f}-{TEST_CI['f1'][1]:.4f}); balanced accuracy {test_metrics['Balanced_Acc']*100:.2f}%
* Newcastle-disease recall {test_metrics['Recall_Newcastle_Disease']*100:.2f}% ; ECE {ece:.4f}

## 4. Why numbers can be lower than published values
{ps_txt}

{papers_md}

## 5. Novelty - poultry vs other-animal rejection
CLIP zero-shot gate (no training data) + Mahalanobis feature-space gate (poultry data only) + confidence gate.
{gate_txt}

## 6. Limitations
Single data source (two Tanzanian regions); farm labels are expert-annotated, not PCR-verified; same-session/same-bird photos cannot be detected without metadata;
CLIP cannot reliably separate chicken from turkey/duck droppings; gate rejection rates are only as valid as the non-poultry evaluation images; no prospective field study.
Predictions must assist, not replace, veterinary diagnosis.
"""
rp = os.path.join(CONFIG['experiment_output_root'], 'reports', 'FINAL_PROJECT_REPORT.md'); open(rp, 'w').write(report)
json.dump({k: (v if not isinstance(v, np.generic) else v.item()) for k, v in CONFIG.items()}, open(os.path.join(CONFIG['experiment_output_root'], 'reports', 'experiment_configuration.json'), 'w'), indent=2, default=str)
print(report); print('[SAVED]', rp)